# Source deletion

Removes sources from the website: deletes the `sources/{ID}/` directory and strips
the corresponding homepage card out of `index.html`. Reverse of `source_setup.ipynb`'s
`process_source()`.

`DRY_RUN = True` by default -- run once to see what *would* happen, then flip to
`False` and rerun to actually delete. Safe to rerun: IDs not found are just reported
and skipped.

In [ ]:
import os
import re
import shutil

In [ ]:
SOURCES_BASE_DIRECTORY = (
    "/home/u7/aakhtarkavan/git_projects/"
    "Galaxy_Kinematics/sources"
)

INDEX_PATH = "../index.html"

In [ ]:
SOURCES_TO_DELETE = [
    # e.g. 12345,
]

DRY_RUN = True  # flip to False once the printout below looks right

In [ ]:
# Each homepage card is a single self-contained block generated by process_source() in
# source_setup.ipynb:
#
#     <div class="col-md-4 gk-card" data-id="{ID}" ...>
#         ...
#     </div>
#
# Every line *inside* the card is indented 12+ spaces (the <a>, inner card div, etc.),
# so the outer closing tag is the only line at exactly 8 spaces -- that's what anchors
# the end of the match. \n* at the front also eats the blank line(s) left behind by
# source_setup.ipynb's `marker + "\n\n" + card` insertion, so deleting a card doesn't
# leave a stack of blank lines between its neighbors.
def make_card_pattern(ID):
    return re.compile(
        r'\n*[ \t]*<div class="col-md-4 gk-card" data-id="%d".*?\n {8}</div>\n' % ID,
        re.DOTALL,
    )


def delete_source(ID, dry_run=True):
    source_dir = f"{SOURCES_BASE_DIRECTORY}/{ID}"
    dir_exists = os.path.isdir(source_dir)

    with open(INDEX_PATH, "r") as f:
        content = f.read()

    new_content, n_removed = make_card_pattern(ID).subn("\n", content)

    if n_removed == 0 and not dir_exists:
        print(f"ID {ID}: not found in index.html or sources/ -- nothing to do.")
        return

    if n_removed == 0:
        print(f"ID {ID}: WARNING -- no card found in index.html, but {source_dir} exists.")
    if not dir_exists:
        print(f"ID {ID}: WARNING -- card found in index.html, but {source_dir} does not exist.")

    if dry_run:
        print(f"ID {ID}: [DRY RUN] would remove card ({n_removed} match) and delete {source_dir}")
        return

    if n_removed > 0:
        with open(INDEX_PATH, "w") as f:
            f.write(new_content)

    if dir_exists:
        shutil.rmtree(source_dir)

    print(f"ID {ID}: removed card ({n_removed} match) and deleted {source_dir}")

In [ ]:
for ID in SOURCES_TO_DELETE:
    delete_source(ID, dry_run=DRY_RUN)